# 119番通報 緊急度判定プロトコル — 遷移をたどってトリアージを機械的に取得

`transition_diagram/protocol.yaml` の遷移グラフを再帰的にたどり、症候別プロトコル毎に
「どの回答列で、どの最終トリアージに到達するか」を全列挙する。

## ルール

- 各 choice の `triage` を読む。`accumulate_triage: true` の場合は値を保持しつつ `next` に進む。
- `triage` 単独（`next` なし）は終端。
- `route_to_protocol` / `route_to` は当該プロトコルとしてはそこで終端（外部ルーティング）。
- `metadata_only` / `transition_only` は素通り（情報収集ノード）。
- どこにも `triage` がつかず終端した場合は protocol の `fallback` を適用:
  - 回答に「不明」(c) が含まれる → `if_any_unknown`
  - そうでなければ → `if_all_symptom_questions_negative`
- 経路上で蓄積された triage と終端 triage の **最重症** をその経路の最終トリアージとする。

重症度順: **R1 > R2 > R3 > Y1 > Y2 > G**

## 1. セットアップ — YAML を読み込んでノード辞書を作る

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

YAML_PATH = Path('protocol.yaml')
if not YAML_PATH.exists():
    YAML_PATH = Path('transition_diagram') / 'protocol.yaml'
with YAML_PATH.open(encoding='utf-8') as f:
    data = yaml.safe_load(f)

labels = data['labels']
entry_flow = {n['id']: n for n in data['entry_flow']}
common_vitals = {n['id']: n for n in data['common_vitals']}
protocols = {p['id']: p for p in data['protocols']}

# protocol id -> {node id -> node}
proto_node_maps = {pid: {n['id']: n for n in p.get('nodes', [])} for pid, p in protocols.items()}

def lookup_node(nid, protocol_id=None):
    if protocol_id and nid in proto_node_maps.get(protocol_id, {}):
        return proto_node_maps[protocol_id][nid]
    if nid in entry_flow:
        return entry_flow[nid]
    if nid in common_vitals:
        return common_vitals[nid]
    for pid, nodes in proto_node_maps.items():
        if nid in nodes:
            return nodes[nid]
    return None

TRIAGE_PRIORITY = {'R1': 6, 'R2': 5, 'R3': 4, 'Y1': 3, 'Y2': 2, 'G': 1}

def max_triage(triages):
    valid = [t for t in triages if t in TRIAGE_PRIORITY]
    return max(valid, key=lambda t: TRIAGE_PRIORITY[t]) if valid else None

print(f'YAML: {YAML_PATH.resolve()}')
print(f'プロトコル数: {len(protocols)} / 共通バイタル: {len(common_vitals)} / entry_flow: {len(entry_flow)}')
print('ラベル:')
for k, v in labels.items():
    print(f'  {k}: {v["category"]} - {v["description"][:30]}...')

## 2. パス列挙ロジック (`enumerate_paths`)

あるプロトコルの開始ノードから全分岐を DFS で列挙し、各パスについて
`final_triage` を計算する。

In [ ]:
def enumerate_paths(protocol_id, *, include_followups=False, max_depth=200):
    proto = protocols[protocol_id]
    proto_nodes = proto_node_maps[protocol_id]
    fallback = proto.get('fallback', {})
    followups = set(proto.get('terminal_followup_chain') or [])
    paths = []

    def get(nid):
        return proto_nodes.get(nid) or lookup_node(nid)

    def finalize(steps, accumulated, orals, *, terminal_triage=None,
                 route_to_protocol=None, route_to=None, fallback_reason=None):
        all_tri = list(accumulated)
        if terminal_triage:
            all_tri.append(terminal_triage)
        final = max_triage(all_tri)
        paths.append({
            'steps': steps,
            'accumulated_triages': list(accumulated),
            'terminal_triage': terminal_triage,
            'final_triage': final,
            'oral_instructions': orals,
            'route_to_protocol': route_to_protocol,
            'route_to': route_to,
            'fallback_used': fallback_reason is not None,
            'fallback_reason': fallback_reason,
        })

    def walk(nid, steps, accumulated, orals, depth):
        if depth > max_depth:
            finalize(steps + [(nid, None, 'MAX_DEPTH')], accumulated, orals,
                     fallback_reason='depth_limit')
            return
        if not include_followups and nid in followups:
            return  # 後続の info チェーンには入らない
        node = get(nid)
        if node is None:
            finalize(steps + [(nid, None, 'UNRESOLVED')], accumulated, orals,
                     fallback_reason='node_not_found')
            return

        # ノード自体が triage を持つ（例: dyspnea_terminal）
        if node.get('triage') and not node.get('choices'):
            t = node['triage']
            o = list(orals) + (node.get('oral_instruction') or [])
            finalize(steps + [(nid, '(node)', node.get('question', ''))],
                     accumulated, o, terminal_triage=t)
            return

        # 情報収集ノードは素通り
        if node.get('metadata_only') or node.get('transition_only'):
            nxt = node.get('next')
            if nxt:
                walk(nxt, steps + [(nid, '(info)', node.get('question', ''))],
                     accumulated, orals, depth + 1)
            else:
                ftri = fallback.get('if_all_symptom_questions_negative') or fallback.get('if_any_unknown')
                finalize(steps + [(nid, '(info)', node.get('question', ''))],
                         accumulated, orals,
                         terminal_triage=ftri,
                         fallback_reason='info_terminal' if ftri else None)
            return

        choices = node.get('choices') or []
        if not choices:
            nxt = node.get('next')
            if nxt:
                walk(nxt, steps + [(nid, None, node.get('question', ''))],
                     accumulated, orals, depth + 1)
            else:
                finalize(steps + [(nid, None, node.get('question', ''))], accumulated, orals)
            return

        for ch in choices:
            ccode = ch.get('code', '')
            ctext = ch.get('text') or ch.get('value') or ''
            new_steps = steps + [(nid, ccode, ctext)]
            new_orals = list(orals) + (ch.get('oral_instruction') or [])
            ch_tri = ch.get('triage')
            accumulate = bool(ch.get('accumulate_triage'))

            if ch.get('route_to_protocol'):
                accs = list(accumulated) + ([ch_tri] if ch_tri else [])
                finalize(new_steps, accs, new_orals,
                         terminal_triage=ch_tri,
                         route_to_protocol=ch['route_to_protocol'])
                continue
            if ch.get('route_to'):
                accs = list(accumulated) + ([ch_tri] if ch_tri else [])
                finalize(new_steps, accs, new_orals,
                         terminal_triage=ch_tri, route_to=ch['route_to'])
                continue

            nxt = ch.get('next')
            if nxt:
                if ch_tri and accumulate:
                    walk(nxt, new_steps, list(accumulated) + [ch_tri], new_orals, depth + 1)
                elif ch_tri:
                    # next もあるが triage が確定しているなら終端優先
                    finalize(new_steps, accumulated, new_orals, terminal_triage=ch_tri)
                else:
                    walk(nxt, new_steps, accumulated, new_orals, depth + 1)
            elif ch_tri:
                finalize(new_steps, accumulated, new_orals, terminal_triage=ch_tri)
            else:
                # PDFセル空欄 → fallback 解決
                is_unknown = (ccode == 'c') or ('不明' in (ctext or ''))
                if is_unknown:
                    ftri = fallback.get('if_any_unknown')
                    reason = 'if_any_unknown'
                else:
                    ftri = fallback.get('if_all_symptom_questions_negative') or \
                           fallback.get('if_no_abdominal_pain_and_underwear_blood_only')
                    reason = 'if_all_symptom_questions_negative'
                finalize(new_steps, accumulated, new_orals,
                         terminal_triage=ftri, fallback_reason=reason)

    start = proto.get('start_node')
    if not start:
        nl = proto.get('nodes') or []
        start = nl[0]['id'] if nl else None
    if start:
        walk(start, [], [], [], 0)
    return paths

## 3. デモ — 1 プロトコルの全パスを列挙

`PROTOCOL_ID` を書き換えれば他のプロトコルも見られる。

In [ ]:
PROTOCOL_ID = 'dyspnea'  # 'palpitation', 'headache', 'numbness', 'trauma' なども

paths = enumerate_paths(PROTOCOL_ID)
print(f'プロトコル: {protocols[PROTOCOL_ID]["name"]} ({PROTOCOL_ID})')
print(f'総パス数: {len(paths)}')
print(f'トリアージ分布: {Counter(p["final_triage"] for p in paths)}')
print()

def short_step(s, n=30):
    return (s[:n - 1] + '…') if len(s) > n else s

def print_path(idx, p):
    print(f'--- パス {idx + 1} ---')
    for nid, ccode, ctext in p['steps']:
        print(f'  [{nid}] -> {ccode}: {short_step(ctext, 50)}')
    extras = []
    if p['accumulated_triages']:
        extras.append(f'蓄積={p["accumulated_triages"]}')
    if p['terminal_triage']:
        extras.append(f'終端={p["terminal_triage"]}')
    if p['route_to_protocol']:
        extras.append(f'route→{p["route_to_protocol"]}')
    if p['fallback_used']:
        extras.append(f'fallback({p["fallback_reason"]})')
    if p['oral_instructions']:
        extras.append(f'口頭指導={p["oral_instructions"]}')
    print(f'  => 最終トリアージ: {p["final_triage"]}  ' + ' '.join(extras))
    print()

for i, p in enumerate(paths[:10]):
    print_path(i, p)
if len(paths) > 10:
    print(f'... 残り {len(paths) - 10} パス省略')

## 4. プロトコル別 トリアージ分布サマリ

全プロトコルについて、列挙したパス数と最終トリアージの内訳を表形式で表示。

In [ ]:
TRIAGE_ORDER = ['R1', 'R2', 'R3', 'Y1', 'Y2', 'G', None]

rows = []
for pid, proto in protocols.items():
    ps = enumerate_paths(pid)
    cnt = Counter(p['final_triage'] for p in ps)
    routed = sum(1 for p in ps if p['route_to_protocol'])
    rows.append({
        'protocol': pid,
        'name': proto.get('name', ''),
        'paths': len(ps),
        **{t or 'unk': cnt.get(t, 0) for t in TRIAGE_ORDER},
        'routed_out': routed,
    })

# 表形式表示（pandas があれば DataFrame、なければ素のテーブル）
try:
    import pandas as pd
    df = pd.DataFrame(rows)
    display(df)
except ImportError:
    cols = list(rows[0].keys())
    widths = {c: max(len(str(c)), max(len(str(r[c])) for r in rows)) for c in cols}
    header = ' | '.join(c.ljust(widths[c]) for c in cols)
    print(header)
    print('-' * len(header))
    for r in rows:
        print(' | '.join(str(r[c]).ljust(widths[c]) for c in cols))

## 5. プロトコル別「到達可能なトリアージ集合」

各プロトコルから理論的にどのトリアージへ落ちうるかを集合で見る。

In [ ]:
def order_key(t):
    return TRIAGE_PRIORITY.get(t, -1)

for pid, proto in protocols.items():
    ps = enumerate_paths(pid)
    triages = sorted({p['final_triage'] for p in ps if p['final_triage']},
                     key=order_key, reverse=True)
    routes = sorted({p['route_to_protocol'] for p in ps if p['route_to_protocol']})
    extra = f'  [外部ルート: {", ".join(routes)}]' if routes else ''
    print(f'{pid:35s} ({proto["name"]:20s}) -> {triages}{extra}')

## 6. 指定した回答列でトリアージをトレース (`trace_path`)

プロトコル ID と `(node_id, choice_code)` のリストを与えると、
その経路を 1 本だけ進めて最終トリアージを返す。

In [ ]:
def trace_path(protocol_id, answers, *, start_node=None):
    proto = protocols[protocol_id]
    proto_nodes = proto_node_maps[protocol_id]
    fallback = proto.get('fallback', {})
    answer_map = dict(answers)
    steps, accumulated, orals = [], [], []

    current = start_node or proto.get('start_node') or (
        proto.get('nodes', [{}])[0].get('id') if proto.get('nodes') else None)

    while current:
        node = proto_nodes.get(current) or lookup_node(current)
        if node is None:
            return {'error': f'node not found: {current}', 'steps': steps}

        if node.get('triage') and not node.get('choices'):
            t = node['triage']
            orals.extend(node.get('oral_instruction') or [])
            steps.append((current, '(node)', node.get('question', ''), t))
            return _wrap(steps, accumulated, orals, terminal=t)

        if node.get('metadata_only') or node.get('transition_only'):
            steps.append((current, '(info)', node.get('question', ''), None))
            current = node.get('next')
            continue

        choices = node.get('choices') or []
        if not choices:
            current = node.get('next')
            continue

        wanted = answer_map.get(current)
        if wanted is None:
            return {'error': f'answer missing for node {current}',
                    'available_choices': [(c.get('code'), c.get('text')) for c in choices],
                    'steps': steps}
        ch = next((c for c in choices if c.get('code') == wanted), None)
        if ch is None:
            return {'error': f'choice {wanted!r} not in {current}',
                    'available_choices': [c.get('code') for c in choices],
                    'steps': steps}

        ch_tri = ch.get('triage')
        steps.append((current, wanted, ch.get('text', ''), ch_tri))
        orals.extend(ch.get('oral_instruction') or [])

        if ch.get('route_to_protocol'):
            return _wrap(steps, accumulated, orals,
                         terminal=ch_tri, route_to_protocol=ch['route_to_protocol'])
        if ch.get('route_to'):
            return _wrap(steps, accumulated, orals,
                         terminal=ch_tri, route_to=ch['route_to'])

        accumulate = bool(ch.get('accumulate_triage'))
        nxt = ch.get('next')
        if nxt:
            if ch_tri and accumulate:
                accumulated.append(ch_tri)
                current = nxt
            elif ch_tri:
                return _wrap(steps, accumulated, orals, terminal=ch_tri)
            else:
                current = nxt
        elif ch_tri:
            return _wrap(steps, accumulated, orals, terminal=ch_tri)
        else:
            is_unknown = (wanted == 'c') or ('不明' in (ch.get('text') or ''))
            ftri = fallback.get('if_any_unknown') if is_unknown else \
                   fallback.get('if_all_symptom_questions_negative')
            return _wrap(steps, accumulated, orals,
                         terminal=ftri, fallback_used=True)

    return _wrap(steps, accumulated, orals)

def _wrap(steps, accumulated, orals, *, terminal=None,
          route_to_protocol=None, route_to=None, fallback_used=False):
    all_tri = list(accumulated) + ([terminal] if terminal else [])
    return {
        'steps': steps,
        'accumulated_triages': list(accumulated),
        'terminal_triage': terminal,
        'final_triage': max_triage(all_tri),
        'oral_instructions': orals,
        'route_to_protocol': route_to_protocol,
        'route_to': route_to,
        'fallback_used': fallback_used,
    }

# --- 使用例: dyspnea で「全部いいえ」のパスを進める ---
# dyspnea には独自の choices ノードが無く dyspnea_terminal (node 自体に triage R2) で終わる構造。
# choices を持つプロトコルで試してみる: palpitation
result = trace_path('palpitation', [
    ('palpitation_heart_history', 'b'),   # 心臓既往 いいえ
    ('palpitation_chest_pain', 'a-ii'),   # 胸痛あり 40歳未満 → Y2
])
print('trace_path 結果:')
for nid, code, text, tri in result['steps']:
    print(f'  [{nid}] {code}: {text}  (triage={tri})')
print(f'  => 最終トリアージ: {result["final_triage"]}')
print(f'  詳細: {result}')

## 7. 全パスを CSV / JSON へ書き出し（任意）

`output/all_paths.csv` に protocol × path 単位で書き出す。

In [ ]:
import csv
import json

OUT_DIR = Path('output')
OUT_DIR.mkdir(exist_ok=True)
csv_path = OUT_DIR / 'all_paths.csv'
json_path = OUT_DIR / 'all_paths.json'

all_rows = []
for pid in protocols:
    for i, p in enumerate(enumerate_paths(pid)):
        all_rows.append({
            'protocol': pid,
            'protocol_name': protocols[pid].get('name', ''),
            'path_index': i,
            'depth': len(p['steps']),
            'path': ' -> '.join(f'{nid}={code}' for nid, code, _ in p['steps']),
            'path_texts': ' / '.join((t or '')[:30] for _, _, t in p['steps']),
            'final_triage': p['final_triage'],
            'terminal_triage': p['terminal_triage'],
            'accumulated_triages': ','.join(p['accumulated_triages']),
            'route_to_protocol': p['route_to_protocol'] or '',
            'route_to': p['route_to'] or '',
            'fallback_used': p['fallback_used'],
            'fallback_reason': p['fallback_reason'] or '',
            'oral_instructions': ','.join(p['oral_instructions']),
        })

with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(all_rows[0].keys()))
    w.writeheader()
    w.writerows(all_rows)

with json_path.open('w', encoding='utf-8') as f:
    json.dump(all_rows, f, ensure_ascii=False, indent=2)

print(f'CSV: {csv_path.resolve()}  ({len(all_rows)} 行)')
print(f'JSON: {json_path.resolve()}')